# 03 - Grain Boundary Analysis 

Common Neighbor Analysis (CNA) on bicrystalline Si structures using OVITO. 
For each grain boundary structure logged in 'grain_boundary_runs', we:
1. Load the bicrystal into an OVITO pipeline
2. Run CNA to classify atoms by local structure
3. Identify the disordered boundary region 
4. Visualize interactively with the OVITO viewport widget
5. Update 'disordered_fraction' and boundary width back to SQL

In [2]:
import ovito 
from ovito.io import import_file 
from ovito.modifiers import CommonNeighborAnalysisModifier, ExpressionSelectionModifier, AssignColorModifier
from ovito.vis import Viewport
import sqlite3 
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

WAFER_AREA_CM2 = (5.431 * 10 * 1e-8) ** 2
print(f"OVITO version: {ovito.version_string}")

/opt/homebrew/Caskroom/miniconda/base/envs/wafer-defect/lib/python3.11/site-packages/ovito/_extensions/anari.py:2: UserWarning: Did you accidentally install the OVITO package from the PyPI repository in an Anaconda/Miniconda Python interpreter using the 'pip' command? This will likely lead to conflicts with existing libraries in the Anaconda environment, and import of the OVITO module may fail with an error related to the Qt framework. To fix this, please uninstall the ovito pip package by running 'pip uninstall -y ovito PySide6' and then install the OVITO Anaconda package provided by OVITO GmbH. Visit https://docs.ovito.org/python/introduction/installation.html for further instructions. If you would rather like to ignore this warning message, add the following code to the top of your Python script:

  import warnings
  warnings.filterwarnings('ignore', message='.*OVITO.*PyPI')

  import ovito._extensions.pyscript


OVITO version: 3.15.4


In [19]:
ovito.scene.pipelines.clear()

pipeline_gb = import_file("data/si_bi_angle15deg.extxyz")

# use IdentifyDiamond modifier instead of CNA for Si
from ovito.modifiers import IdentifyDiamondModifier

idd = IdentifyDiamondModifier()
pipeline_gb.modifiers.append(idd)

# color disordered atoms (not cubic diamond) red
pipeline_gb.modifiers.append(ExpressionSelectionModifier(expression="StructureType == 0"))
pipeline_gb.modifiers.append(AssignColorModifier(color=(1, 0, 0)))

data = pipeline_gb.compute()
data.particles.vis.radius = 0.5

n_total = data.particles.count
print("Attributes:", [k for k in data.attributes.keys() if "Diamond" in k or "Identify" in k])
print(f"Total atoms: {n_total}")

ovito.scene.pipelines.append(pipeline_gb)
vp = Viewport()
vp.type = Viewport.Type.Perspective
vp.camera_pos = (50, 50, 50)
vp.camera_dir = (-1, -1, -1)

widget = ovito.vis._viewport_create_jupyter_widget._Viewport_create_jupyter_widget(vp)
widget.layout.width = "400px"
widget.layout.height = "400px"
widget

Attributes: ['IdentifyDiamond.counts.OTHER', 'IdentifyDiamond.counts.CUBIC_DIAMOND', 'IdentifyDiamond.counts.CUBIC_DIAMOND_FIRST_NEIGHBOR', 'IdentifyDiamond.counts.CUBIC_DIAMOND_SECOND_NEIGHBOR', 'IdentifyDiamond.counts.HEX_DIAMOND', 'IdentifyDiamond.counts.HEX_DIAMOND_FIRST_NEIGHBOR', 'IdentifyDiamond.counts.HEX_DIAMOND_SECOND_NEIGHBOR']
Total atoms: 8000


JupyterViewportWidget(layout=Layout(height='400px', width='400px'))

In [15]:
conn = sqlite3.connect("defect_db.sqlite")
rows = conn.execute("SELECT id, structure_file, misorientation_angle FROM grain_boundary_runs").fetchall()

print("Updating grain boundary runs from OVITO IdentifyDiamond analysis...")
for row_id, fname, angle in rows:
    p = import_file(fname)
    idd = IdentifyDiamondModifier()
    p.modifiers.append(idd)
    d = p.compute()

    n_total = d.particles.count
    n_disordered = int(d.attributes["IdentifyDiamond.counts.OTHER"])
    disordered_frac = n_disordered / n_total

    conn.execute(""" 
        UPDATE grain_boundary_runs 
        SET disordered_fraction = ?
        WHERE id = ? 
    """, (disordered_frac, row_id))
    print(f"id: {row_id}, angle: {angle:.2f}, total atoms: {n_total}, disordered atoms: {n_disordered}, disordered fraction: {disordered_frac:.4f}")
conn.commit()
conn.close()
print("\nDB updated.")

Updating grain boundary runs from OVITO IdentifyDiamond analysis...
id: 1, angle: 15.00, total atoms: 8000, disordered atoms: 730, disordered fraction: 0.0912
id: 2, angle: 30.00, total atoms: 8000, disordered atoms: 1960, disordered fraction: 0.2450
id: 3, angle: 45.00, total atoms: 8000, disordered atoms: 2960, disordered fraction: 0.3700
id: 7, angle: 15.00, total atoms: 8000, disordered atoms: 730, disordered fraction: 0.0912
id: 8, angle: 30.00, total atoms: 8000, disordered atoms: 1960, disordered fraction: 0.2450
id: 9, angle: 45.00, total atoms: 8000, disordered atoms: 2960, disordered fraction: 0.3700
id: 10, angle: 15.00, total atoms: 8000, disordered atoms: 730, disordered fraction: 0.0912
id: 11, angle: 30.00, total atoms: 8000, disordered atoms: 1960, disordered fraction: 0.2450
id: 12, angle: 45.00, total atoms: 8000, disordered atoms: 2960, disordered fraction: 0.3700
id: 13, angle: 15.00, total atoms: 8000, disordered atoms: 730, disordered fraction: 0.0912
id: 14, angl

In [16]:
conn = sqlite3.connect("defect_db.sqlite")
conn.execute("DELETE FROM grain_boundary_runs WHERE id > 3")
conn.commit()
print(f"Rows remaining: {conn.execute('SELECT COUNT(*) FROM grain_boundary_runs').fetchone()[0]}")
conn.close()

Rows remaining: 3
